Warung Suara - Synthetic Voice Dataset Generator
==================================================
Tahap 1 dari alur dataset (lihat "ALUR DATASET" di bawah). Men-generate
pasangan (audio, label terstruktur) yang dipakai sebagai training data untuk
fine-tuning Whisper pada tahap berikutnya.

ALUR DATASET (untuk proposal - bagian Metodologi):
  1. Sumber data: SINTETIK, bukan rekaman asli. Dipilih karena (a) belum ada
     dataset publik voice-note transaksi warung Bahasa Indonesia dengan label
     terstruktur (item/qty/unit/action), dan (b) rulebook AIC mengizinkan
     dataset sintetik selama alur perolehan & preprocessing-nya dijelaskan.
  2. Skema entitas (item, qty, unit, action) didefinisikan lebih dulu, lalu
     kalimat dibuat dari kombinasi template x vocab -> ini yang menjamin
     SETIAP audio yang dihasilkan otomatis punya ground-truth label yang
     akurat (tidak perlu anotasi manual/transkripsi ulang).
  3. Variasi disengaja dimasukkan di 3 sumbu: gaya kalimat (TEMPLATES),
     bentuk pengucapan angka (QTY_WORDS: kata vs digit, termasuk "se-"),
     dan sinonim aksi (ACTIONS: "laku"/"kejual"/"terjual", dst) - supaya
     model tidak overfit ke satu pola ucapan.
  4. Rendering suara: tiap kalimat teks diucapkan lewat model TTS pretrained
     CSM-1B Indonesian (lihat catatan "PENGGUNAAN MODEL PRETRAINED" di bawah),
     dengan speaker_id diacak (0-80) tiap sampel supaya dataset punya variasi
     81 karakter suara berbeda dan model STT nantinya tidak overfit ke 1 suara.
  5. Output: audio .wav + manifest berlabel (JSONL & CSV) -> manifest ini
     yang jadi INPUT LANGSUNG ke script fine-tuning Whisper di tahap
     berikutnya (kolom "text" jadi target transkripsi, kolom lain jadi
     ground-truth untuk evaluasi extractor rule-based/NLP).

PENGGUNAAN MODEL PRETRAINED (untuk kepatuhan rulebook AIC):
  Model `Ellbendls/csm-1b-indonesian-fine-tuned` di sini dipakai APA ADANYA
  (inference only, tanpa fine-tune) HANYA untuk membangkitkan data suara
  sintetik - bukan komponen inti produk Warung Suara. Komponen inti yang
  wajib di-fine-tune sesuai rulebook (poin 10 Ketentuan Khusus: "Model wajib
  di fine tune sesuai dengan inovasi fitur per tim") adalah model Whisper
  STT yang di-fine-tune pakai dataset hasil script ini - dilakukan di
  script terpisah (tahap Minggu 2), bukan di sini.

CATATAN PENTING:
- Script ini butuh akses ke huggingface.co untuk download model & GPU untuk
  kecepatan wajar (CPU bisa jalan tapi lambat). Jalankan di Colab/lokal,
  BUKAN di sandbox ini (jaringan saya di sini tidak bisa reach huggingface.co).
- Sesuaikan MODEL_ID, jumlah sampel, dan vocab item sesuai kebutuhan.
- Simpan juga versi awal manifest.jsonl/csv di repo (atau ringkasannya) -
  ini bukti "alur memperoleh dataset" yang diminta proposal & bisa dicek
  panitia saat cross-check commit history.

Install dulu:
    pip install transformers torch soundfile

In [3]:
import os
import json
import random
import csv
from pathlib import Path

import torch
import soundfile as sf
from transformers import CsmForConditionalGeneration, AutoProcessor

# 1. KONFIGURASI

In [4]:
MODEL_ID = "Ellbendls/csm-1b-indonesian-fine-tuned"
OUTPUT_DIR = Path("dataset_warung_suara")
AUDIO_DIR = OUTPUT_DIR / "audio"
NUM_SPEAKERS = 81          # speaker_id valid: 0-80
SAMPLE_RATE = 24000
MAX_NEW_TOKENS = 125       # ~10 detik audio, cukup untuk kalimat pendek
SAMPLES_PER_TEMPLATE_COMBO = 2   # berapa kali tiap kombinasi diucap ulang (speaker beda)
SEED = 42

random.seed(SEED)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# 2. VOCAB & SKEMA ENTITAS

In [5]:
# action: "keluar" (barang laku/terjual) atau "masuk" (restock)
ACTIONS = {
    "keluar": ["laku", "kejual", "terjual", "abis dibeli", "keluar"],
    "masuk":  ["masuk", "stok masuk", "baru dateng", "restock", "nambah stok"],
}

# item -> unit default yang natural dipakai pedagang
ITEMS = {
    "indomie":        "bungkus",
    "telor":          "kg",
    "gula":           "kg",
    "beras":          "kg",
    "minyak goreng":  "liter",
    "kecap":          "botol",
    "sabun mandi":    "batang",
    "rokok":          "bungkus",
    "aqua gelas":     "dus",
    "gas elpiji":     "tabung",
    "kopi sachet":    "renceng",
    "susu kental manis": "kaleng",
}

# angka diucapkan dalam berbagai bentuk (digit & kata) biar model robust
QTY_WORDS = {
    1: ["satu", "se", "1"],
    2: ["dua", "2"],
    3: ["tiga", "3"],
    5: ["lima", "5"],
    10: ["sepuluh", "10"],
}

# 3. TEMPLATE KALIMAT (variasi gaya ngomong pedagang)

In [6]:
TEMPLATES = [
    "{action} {item} {qty} {unit}",
    "{item} {action} {qty} {unit}",
    "tadi {action} {item} {qty} {unit}",
    "{action} {item} {qty} {unit} ya",
    "eh {item} {action} {qty} {unit}",
    "barusan {action} {item} {qty} {unit}",
    "{qty} {unit} {item} {action}",
]


def build_dataset_entries():
    """
    Preprocessing tahap 1: kombinasi (item x qty x action) -> teks + label.

    Tiap entry punya "text" (kalimat yang akan diucapkan TTS) dan "label"
    (ground-truth terstruktur: item, qty, unit, action). Karena label
    dibuat BERSAMAAN dengan teks (bukan diekstrak belakangan), tidak ada
    risiko label salah/hasil anotasi manual yang bias.
    """
    entries = []
    for item, unit in ITEMS.items():
        for qty, qty_variants in QTY_WORDS.items():
            for action_type, action_variants in ACTIONS.items():
                template = random.choice(TEMPLATES)
                qty_word = random.choice(qty_variants)
                action_word = random.choice(action_variants)

                # "se" khusus butuh nyambung ke unit (se + kilo, bukan "se kilo")
                unit_text = unit
                if qty_word == "se":
                    unit_text = "kilo" if unit == "kg" else unit

                text = template.format(
                    action=action_word, item=item, qty=qty_word, unit=unit_text
                ).strip()

                entries.append({
                    "text": text,
                    "label": {
                        "item": item,
                        "qty": qty,
                        "unit": unit,
                        "action": action_type,
                    },
                })
    return entries


# 4. LOAD MODEL TTS

In [7]:
def load_tts():
    """
    Load model TTS pretrained (inference only, tidak di-fine-tune).
    Dipakai murni sebagai "mesin perekam suara sintetik" untuk membangkitkan
    dataset - bukan bagian dari arsitektur inti produk Warung Suara.
    """
    print(f"Loading model {MODEL_ID} ...")
    model = CsmForConditionalGeneration.from_pretrained(MODEL_ID)
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"Model loaded on {device}")
    return model, processor, device


def synthesize(model, processor, device, text: str, speaker_id: int) -> "torch.Tensor":
    """
    Render 1 kalimat teks -> audio, dengan speaker_id acak (0-80) supaya
    dataset akhir punya variasi karakter suara (mencegah model STT nanti
    overfit ke satu jenis suara/nada).
    """
    prompt = f"[{speaker_id}]{text}"
    inputs = processor(prompt, add_special_tokens=True).to(device)
    audio_values = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        depth_decoder_top_p=0.9,
        depth_decoder_do_sample=True,
        depth_decoder_temperature=0.8,
        output_audio=True,
    )
    return audio_values[0].to(torch.float32).cpu().numpy()


# 5. MAIN GENERATION LOOP

In [8]:
def main():
    entries = build_dataset_entries()
    print(f"Total kombinasi kalimat unik: {len(entries)}")
    print(f"Total sampel audio yang akan dibuat: {len(entries) * SAMPLES_PER_TEMPLATE_COMBO}")

    model, processor, device = load_tts()

    manifest = []
    idx = 0
    for entry in entries:
        for _ in range(SAMPLES_PER_TEMPLATE_COMBO):
            speaker_id = random.randint(0, NUM_SPEAKERS - 1)
            filename = f"sample_{idx:05d}.wav"
            filepath = AUDIO_DIR / filename

            try:
                audio = synthesize(model, processor, device, entry["text"], speaker_id)
                sf.write(str(filepath), audio, SAMPLE_RATE)
            except Exception as e:
                print(f"  [skip] gagal generate '{entry['text']}' (speaker {speaker_id}): {e}")
                continue

            manifest.append({
                "audio_path": str(filepath.as_posix()),
                "text": entry["text"],
                "speaker_id": speaker_id,
                **entry["label"],
            })

            idx += 1
            if idx % 25 == 0:
                print(f"  {idx} sampel selesai...")

    # simpan manifest JSONL (mudah dipakai untuk fine-tuning Whisper)
    jsonl_path = OUTPUT_DIR / "manifest.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in manifest:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    # simpan juga CSV (enak dilihat manual / dibuka Excel)
    csv_path = OUTPUT_DIR / "manifest.csv"
    if manifest:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
            writer.writeheader()
            writer.writerows(manifest)

    print(f"\nSelesai. {len(manifest)} sampel audio+label tersimpan di '{OUTPUT_DIR}/'")
    print(f"  - Audio  : {AUDIO_DIR}/")
    print(f"  - Label  : {jsonl_path} & {csv_path}")


if __name__ == "__main__":
    main()


Total kombinasi kalimat unik: 120
Total sampel audio yang akan dibuat: 240
Loading model Ellbendls/csm-1b-indonesian-fine-tuned ...


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2050), got 128002. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2050), got 128000. This may result in unexpected behavior.
[transformers] Model config: audio_token_id must be `None` or an integer within the vocabulary (between 0 and 2050), got 128002. This may result in unexpected behavior.
[transformers] Model config: audio_eos_token_id must be `None` or an integer within the vocabulary (between 0 and 2050), got 128003. This may result in unexpected behavior.


OSError: Ellbendls/csm-1b-indonesian-fine-tuned does not appear to have a file named pytorch_model.bin or model.safetensors.